In [0]:
CREATE TABLE IF NOT EXISTS sac.customer.churn (
		customer_id STRING NOT NULL,
		churned BOOLEAN,
		churn_reason STRING,
		CONSTRAINT churn_pk PRIMARY KEY (customer_id),
		CONSTRAINT churn_fk FOREIGN KEY (customer_id) REFERENCES sac.customer.customer (customer_id)
	);

MERGE INTO
	sac.customer.churn s
USING (
	SELECT
		c.customer_id,
		CASE
			WHEN c.churned = 1 THEN TRUE
			ELSE FALSE
		END AS churned,
		c.churn_reason
	FROM
		sac.customer.churn_bronze c
	QUALIFY
		row_number() OVER (PARTITION BY c.customer_id ORDER BY c.ingestion_time DESC) = 1
) b
ON
	b.customer_id = s.customer_id
WHEN MATCHED AND
	sha1(concat_ws('|', b.churned, b.churn_reason))
		!= sha1(concat_ws('|', s.churned, s.churn_reason))
	THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;